# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

Both findings are from `docs/flyrank-seo-research-march-2026.pdf`, ML Appendix.

**Finding A — "What Predicts Health?" (Random Forest feature importance, p.27).** Reports Average Position (43%), Impressions (32%), and Scroll Depth (15%) as the top predictors of `health_score`. **Where the label comes from:** the paper's own "How to Read This Paper" section (p.5) defines `health_score = Impressions(30pts) + Position(30pts) + CTR(20pts) + ScrollDepth(20pts)`. Three of the top four "predictors" (Position, Impressions, Scroll Depth — 80 of the 100 possible points) are literal components of the label, not independent signals. **Does the validation design carry the claim?** Partially — the paper does add the caveat "importance is descriptive rather than causal" — but a holdout-tested model doesn't fix this: a model can hold out perfectly on a label made of the model's own inputs. The paper never runs the with/without test that would show how much of that 90% combined importance is definitional vs. genuinely new signal. I ran that test myself below on an analog built from my own dataset.

**Finding B — "What Predicts Growth?" (Logistic regression, 71% holdout accuracy, p.29).** **Where the label comes from:** `trend_direction`, computed from the 30-day-vs-previous-30-day impression change (p.5) — not derived from the features used to predict it, so this one is clean on the label side. **Does the validation design carry the claim?** Two gaps: (1) the methodology page (p.36) states an 80/20 split with no mention of grouping by brand, even though 57 brands is exactly the kind of repeating entity the leakage/validation playbook says to group by — pages under one brand share a CMS, an editorial team, a template. (2) **no base rate is reported anywhere next to the 71% figure** — nothing in the paper says what share of the sampled pages were growing vs. declining, so a reader can't tell if 71% is real skill or close to what guessing the majority class would give for free. I tested both concerns on an analog below.

## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

My Week-5/8 model (`w05_model.ipynb`) was already built with a client-grouped split — the "after" here. So this section runs the honest counterfactual for Finding B directly: the same logistic regression, same features, same label idea (growth vs. decline), once under a random 80/20 split (the paper's stated method) and once under a client-grouped split, to see whether the choice would have mattered on data shaped like mine.

In [1]:
import pandas as pd
import numpy as np
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score

df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Growth-vs-decline analog of Finding B, using trend_direction (up/down only, matching the paper's framing)
d = df[df.trend_direction.isin(["up", "down"])].copy()
d["is_growing"] = (d.trend_direction == "up").astype(int)

# Feature set deliberately excludes the 30d/prev30d windows that build trend_direction itself --
# same leakage-taxonomy-2 rule from ML-04/ML-05.
feats = ["content_age_days", "days_since_last_update", "days_with_impressions", "avg_position",
         "word_count", "impressions_90d", "search_volume", "clicks_90d", "ai_sessions_90d", "sessions_90d"]
for c in feats:
    d[c] = d[c].fillna(d[c].median())
X = d[feats + ["client_id", "is_growing"]].copy()
print(f"n={len(X)}, {X.client_id.nunique()} clients, base rate (growing)={X.is_growing.mean():.3f}")

# BEFORE: random 80/20 split -- mirrors the paper's stated method
Xtr, Xte, ytr, yte = train_test_split(X[feats], X["is_growing"], test_size=0.2, random_state=42, stratify=X["is_growing"])
scaler = StandardScaler()
clf = LogisticRegression(max_iter=2000, random_state=42).fit(scaler.fit_transform(Xtr), ytr)
acc_random = accuracy_score(yte, clf.predict(scaler.transform(Xte)))
base_random = max(yte.mean(), 1 - yte.mean())

# AFTER: client-grouped 80/20 split -- the honest version
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=42)
tr_idx, te_idx = next(gss.split(X, groups=X["client_id"]))
Xtr2, Xte2 = X.iloc[tr_idx][feats], X.iloc[te_idx][feats]
ytr2, yte2 = X.iloc[tr_idx]["is_growing"], X.iloc[te_idx]["is_growing"]
scaler2 = StandardScaler()
clf2 = LogisticRegression(max_iter=2000, random_state=42).fit(scaler2.fit_transform(Xtr2), ytr2)
acc_grouped = accuracy_score(yte2, clf2.predict(scaler2.transform(Xte2)))
base_grouped = max(yte2.mean(), 1 - yte2.mean())

print(f"\nBEFORE (random split):  accuracy={acc_random:.3f}  base_rate={base_random:.3f}  "
      f"skill_above_base={acc_random-base_random:+.3f}")
print(f"AFTER  (grouped split): accuracy={acc_grouped:.3f}  base_rate={base_grouped:.3f}  "
      f"skill_above_base={acc_grouped-base_grouped:+.3f}")

n=20650, 30 clients, base rate (growing)=0.212

BEFORE (random split):  accuracy=0.790  base_rate=0.787  skill_above_base=+0.002
AFTER  (grouped split): accuracy=0.759  base_rate=0.769  skill_above_base=-0.010


**Both numbers, read honestly:** on this analog, accuracy barely differs between the two splits (0.790 vs. 0.759) — so on my dataset, the grouping choice alone wasn't the main problem. The bigger issue is what it's being compared against: the base rate in both splits (0.787, 0.769) is essentially the same as the model's accuracy, meaning **skill above baseline is ~0 or slightly negative in both cases.** This directly supports the Finding-B critique: without a printed base rate, a 71%-shaped number can look like real skill when it might just be the majority class.

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

**Part 1 — reconfirm my own final feature set (unchanged since ML-05/ML-08) is still clean.**

In [2]:
pool = df[(df.impressions_90d >= 500) & (df.avg_position > 0) & (df.avg_position <= 20)].copy()
pool["tier_median_ctr"] = pool.groupby("position_tier")["ctr"].transform("median")

numeric_feats = ["search_volume", "competition", "cpc", "word_count", "char_count",
    "content_age_days", "days_since_last_update", "engagement_rate", "scroll_rate",
    "ai_traffic_pct", "ai_sessions_90d", "sessions_90d", "pageviews_90d", "users_90d",
    "engaged_sessions_90d", "scroll_events_90d", "days_with_impressions", "days_with_sessions"]
cat_feats = ["content_type", "main_intent", "competition_level",
             "word_count_tier", "char_count_tier", "age_tier", "freshness_tier"]
final_feature_cols = numeric_feats + [f"has_{c}" for c in
    ["search_volume", "competition", "cpc", "word_count", "char_count"]] + cat_feats  # pre-one-hot names

banned = {"ctr", "clicks_90d", "trend_direction", "trend_pct",
          "impressions_last_30d", "clicks_last_30d", "sessions_last_30d",
          "impressions_prev_30d", "clicks_prev_30d", "sessions_prev_30d",
          "provider_used", "model_used", "content_id", "client_id",
          "avg_position", "position_tier", "impression_tier", "impressions_90d"}
overlap = banned & set(final_feature_cols)
print("Banned fields in my final feature set:", overlap or "none")
assert not overlap, "Leakage regression -- an excluded field crept back into the model."
print("Reconfirmed clean -- same 25 base features carried through ML-05 -> ML-08 -> capstone.")

Banned fields in my final feature set: none
Reconfirmed clean -- same 25 base features carried through ML-05 -> ML-08 -> capstone.


**Part 2 — apply the same hunt to the paper's Finding A, since Section 1 flagged it as label-derived (leakage taxonomy #1).** I can't touch FlyRank's real `health_score` (it isn't in my starter CSV), so I built an analog from my own data using the paper's own published formula, then ran the skill's canonical test: train once WITH the suspect components, once WITHOUT.

In [3]:
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import r2_score

d2 = df[df.avg_position > 0].copy()  # avg_position==0 means "no data", can't score position
d2["scroll_rate"] = d2["scroll_rate"].fillna(d2["scroll_rate"].median())

def pct_rank(s):
    return s.rank(pct=True)

# health_score analog, using the paper's own published weights (p.5): Impressions(30) + Position(30) + CTR(20) + ScrollDepth(20)
d2["health_score_proxy"] = (
    pct_rank(d2.impressions_90d) * 30
    + (1 - pct_rank(d2.avg_position)) * 30
    + pct_rank(d2.ctr) * 20
    + pct_rank(d2.scroll_rate) * 20
)

suspects = ["impressions_90d", "avg_position", "ctr", "scroll_rate"]  # the label's own literal components
other_feats = ["content_age_days", "days_since_last_update", "word_count", "search_volume",
               "competition", "cpc", "sessions_90d", "users_90d", "engagement_rate",
               "ai_traffic_pct", "days_with_impressions", "days_with_sessions"]
for c in other_feats:
    d2[c] = d2[c].fillna(d2[c].median())

Xtr, Xte, ytr, yte = train_test_split(d2[suspects+other_feats], d2["health_score_proxy"], test_size=0.2, random_state=42)

rf_with = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf_with.fit(Xtr[suspects+other_feats], ytr)
r2_with = r2_score(yte, rf_with.predict(Xte[suspects+other_feats]))

rf_without = RandomForestRegressor(n_estimators=300, max_depth=8, random_state=42, n_jobs=-1)
rf_without.fit(Xtr[other_feats], ytr)
r2_without = r2_score(yte, rf_without.predict(Xte[other_feats]))

print(f"R^2 WITH the suspects (impressions/position/ctr/scroll):    {r2_with:.3f}")
print(f"R^2 WITHOUT them (other_feats only):                        {r2_without:.3f}")

imp = pd.Series(rf_with.feature_importances_, index=suspects+other_feats).sort_values(ascending=False)
print("\ntop importances WITH suspects (share of total):")
print((imp / imp.sum() * 100).round(1).head(4))

print("\n-> The confession: R^2 collapses from 0.99 to 0.64 the moment the label's own components "
      "are removed, and the with-suspects run reproduces the paper's own ranking almost exactly "
      "(ctr/position/scroll dominate). Finding A's feature-importance chart is mostly measuring "
      "the label's own formula reflected back at itself -- the paper's hedge language is correct "
      "in spirit but understates how large the effect is.")

R^2 WITH the suspects (impressions/position/ctr/scroll):    0.987
R^2 WITHOUT them (other_feats only):                        0.639

top importances WITH suspects (share of total):
ctr                52.2
avg_position       27.6
scroll_rate        14.4
impressions_90d     5.6
dtype: float64

-> The confession: R^2 collapses from 0.99 to 0.64 the moment the label's own components are removed, and the with-suspects run reproduces the paper's own ranking almost exactly (ctr/position/scroll dominate). Finding A's feature-importance chart is mostly measuring the label's own formula reflected back at itself -- the paper's hedge language is correct in spirit but understates how large the effect is.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

**Original (from `w04_baseline_score.ipynb`, Section 1):**
> "The score is the estimated clicks/90d the page is leaving on the table."

That reads as a guaranteed, recoverable number — as if fixing the page returns exactly that many clicks. It's really a gap against a same-tier peer median, computed from one 90-day snapshot, with no causal test that a refresh actually closes the gap.

**Rewritten:**
> "The score is a directional estimate of how many clicks this page might be missing relative to other pages in its own position tier over the same 90 days — a prioritization signal for deciding what to review first, not a guaranteed recoverable click count, and not evidence that a refresh will close the gap."

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.